<a href="https://colab.research.google.com/github/Yisihaq01/breast-histology-calibration/blob/main/Notebooks/07_efficientnet_b0_posthoc_calibration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Stage 7 — Post-hoc Probability Calibration

## Objective

Evaluate whether post-hoc calibration improves the probability reliability of the final EfficientNet-B0 breast histopathology classifier.

The final model was trained using all 49 Development patients for five fixed epochs. Its weights are now frozen.

## Calibration Methods

Two methods will be fitted using the separate Calibration cohort of 16 patients and 382 images at 40× magnification.

1. **Temperature scaling:** Learn a positive temperature that rescales image-level logits without changing their ranking or the 0.5 decision boundary.

2. **Platt scaling:** Learn a logistic transformation with a slope and intercept applied to image-level logits.

The uncalibrated model will serve as the baseline.

Image-level scores will use mean patch logits, consistent with the primary aggregation protocol established during Development.

## Data Isolation

Only the designated Calibration partition will be used to fit calibration parameters.

The final CNN weights will not be updated.

No independent Test data, magnification-shift Test images, or external datasets will be used to fit or select calibrators.

Calibration-cohort metrics are descriptive fitting diagnostics, not independent generalization results. The fitted calibrators will be frozen before subsequent Test and shift evaluations.


In [ ]:

# ============================================================
# Environment, Final Model Provenance, Calibration Metadata
# ============================================================

import json
import subprocess
from pathlib import Path

import pandas as pd
import torch

# ------------------------------------------------------------
# 1. Locate or clone the repository
# ------------------------------------------------------------

REPO_ROOT = Path(
    "/content/breast-histology-calibration"
)

REPO_URL = (
    "https://github.com/Yisihaq01/"
    "breast-histology-calibration.git"
)

if not REPO_ROOT.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_ROOT)],
        check=True
    )
    print("Repository cloned.")
else:
    print("Existing repository located.")

assert (REPO_ROOT / ".git").is_dir()

commit = subprocess.check_output(
    [
        "git", "-C", str(REPO_ROOT),
        "rev-parse", "--short", "HEAD"
    ],
    text=True
).strip()

assert torch.cuda.is_available(), (
    "Select a GPU runtime in Colab."
)

DEVICE = torch.device("cuda")

print("\n" + "=" * 70)
print("STAGE 7 — ENVIRONMENT")
print("=" * 70)

print("Repository:", REPO_ROOT)
print("Git commit:", commit)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 2. Verify Stage 5 architecture-selection decision
# ------------------------------------------------------------

decision_path = (
    REPO_ROOT /
    "results/formal_development/efficientnet_b0/"
    "cross_fold_analysis/stage5_architecture_selection.json"
)

assert decision_path.is_file()

with open(decision_path, encoding="utf-8") as f:
    decision = json.load(f)

assert (
    decision["selected_primary_architecture"]
    == "efficientnet_b0"
)

print("\nPASS: Stage 5 architecture decision verified.")

# ------------------------------------------------------------
# 3. Verify Stage 6 final training records
# ------------------------------------------------------------

STAGE6_RESULTS_DIR = (
    REPO_ROOT /
    "results/final_development/efficientnet_b0"
)

completion_path = (
    STAGE6_RESULTS_DIR /
    "final_training_completion.json"
)

audit_path = (
    STAGE6_RESULTS_DIR /
    "final_model_independent_audit.json"
)

assert completion_path.is_file(), (
    "Stage 6 completion metadata not found. "
    "Check that both Stage 6 GitHub commits "
    "were pushed."
)

assert audit_path.is_file(), (
    "Stage 6 independent audit metadata missing."
)

with open(completion_path, encoding="utf-8") as f:
    completion = json.load(f)

with open(audit_path, encoding="utf-8") as f:
    audit = json.load(f)

EXPECTED_CHECKPOINT_SHA256 = (
    "84ea0871a309f3d68b8958b68b6da4f01adb31d3109fa4e0fc57f6670b63c9bb"
)

assert completion["training_completed"] is True
assert completion["epochs_completed"] == 5
assert completion["checkpoint_selected"] == "fixed_epoch_5"

assert completion["checkpoint_sha256"] == (
    EXPECTED_CHECKPOINT_SHA256
)

assert audit["checkpoint_sha256"] == (
    EXPECTED_CHECKPOINT_SHA256
)

assert audit["independent_model_reconstruction"] is True
assert audit["strict_checkpoint_loading"] is True

FINAL_CHECKPOINT_FILENAME = (
    completion["checkpoint_filename"]
)

print("PASS: Stage 6 final training verified.")
print("PASS: Independent model audit verified.")
print("Checkpoint:", FINAL_CHECKPOINT_FILENAME)
print("Expected SHA-256:", EXPECTED_CHECKPOINT_SHA256)

# ------------------------------------------------------------
# 4. Inventory original frozen partition metadata
# ------------------------------------------------------------

METADATA_DIR = REPO_ROOT / "Metadata"

assert METADATA_DIR.is_dir()

metadata_files = sorted([
    path
    for path in METADATA_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {".csv", ".json"}
])

assert metadata_files

print("\n" + "=" * 70)
print("FROZEN METADATA INVENTORY")
print("=" * 70)

for path in metadata_files:
    print(path.relative_to(REPO_ROOT))

# ------------------------------------------------------------
# 5. Identify candidate Calibration metadata
# ------------------------------------------------------------

candidate_terms = (
    "calibration",
    "partition",
    "split",
    "patient"
)

candidates = [
    path
    for path in metadata_files
    if any(
        term in path.name.lower()
        for term in candidate_terms
    )
]

print("\n" + "=" * 70)
print("CALIBRATION / SPLIT METADATA CANDIDATES")
print("=" * 70)

for path in candidates:

    print("\nFile:", path.relative_to(REPO_ROOT))

    if path.suffix.lower() == ".csv":
        columns = pd.read_csv(
            path,
            nrows=0
        ).columns.tolist()

        print("Columns:", columns)

if not candidates:
    print(
        "No obvious candidate by filename. "
        "We will inspect the frozen metadata structure."
    )

# ------------------------------------------------------------
# 6. Final checks
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STAGE 7 — INITIAL PREFLIGHT")
print("=" * 70)

print("PASS: GPU runtime available.")
print("PASS: GitHub repository located.")
print("PASS: Stage 5 architecture decision verified.")
print("PASS: Stage 6 model provenance verified.")
print("PASS: Final checkpoint identity established.")
print("PASS: Original metadata files inventoried.")
print("PASS: No model training performed.")
print("PASS: No Calibration images loaded yet.")
print("PASS: No Test or external images accessed.")
print("PASS: No calibration parameters fitted.")


Repository cloned.

STAGE 7 — ENVIRONMENT
Repository: /content/breast-histology-calibration
Git commit: f55bcd7
PyTorch: 2.11.0+cu130
GPU: Tesla T4

PASS: Stage 5 architecture decision verified.
PASS: Stage 6 final training verified.
PASS: Independent model audit verified.
Checkpoint: efficientnet_b0_final_epoch5.pt
Expected SHA-256: 84ea0871a309f3d68b8958b68b6da4f01adb31d3109fa4e0fc57f6670b63c9bb

FROZEN METADATA INVENTORY
Metadata/BACH_metadata/bach_audit_summary.json
Metadata/BACH_metadata/bach_binary_label_mapping.json
Metadata/BACH_metadata/bach_duplicate_audit.csv
Metadata/BACH_metadata/bach_image_audit.csv
Metadata/BACH_metadata/bach_patch_audit.csv
Metadata/BACH_metadata/bach_patch_manifest.csv
Metadata/BACH_metadata/bach_preprocessing_config.json
Metadata/BACH_metadata/bach_verified_metadata.csv
Metadata/BACH_metadata/microscopy_ground_truth.csv
Metadata/BACH_metadata/microscopy_ground_truth_with_headers.csv
Metadata/baseline_train_patch_manifest.csv
Metadata/baseline_val_patc

In [ ]:

# ============================================================
# Frozen Calibration Cohort Verification
# ============================================================

# ------------------------------------------------------------
# 1. Load original frozen patient assignments
# ------------------------------------------------------------

SPLIT_PATH = (
    REPO_ROOT / "Metadata/breakhis_patient_split.csv"
)

METADATA_40X_PATH = (
    REPO_ROOT / "Metadata/breakhis_40x_final_metadata.csv"
)

assert SPLIT_PATH.is_file()
assert METADATA_40X_PATH.is_file()

patient_splits = pd.read_csv(
    SPLIT_PATH,
    dtype={"patient_id": str}
)

assert {"patient_id", "split"}.issubset(
    patient_splits.columns
)

assert patient_splits["patient_id"].is_unique

patient_splits["split_normalized"] = (
    patient_splits["split"]
    .astype(str)
    .str.strip()
    .str.lower()
)

print("=" * 70)
print("FROZEN PATIENT PARTITIONS")
print("=" * 70)

print(
    patient_splits["split_normalized"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 2. Identify frozen Calibration patients
# ------------------------------------------------------------

calibration_patient_ids = set(
    patient_splits.loc[
        patient_splits["split_normalized"] == "calibration",
        "patient_id"
    ]
)

development_patient_ids = set(
    patient_splits.loc[
        patient_splits["split_normalized"] == "development",
        "patient_id"
    ]
)

test_patient_ids = set(
    patient_splits.loc[
        patient_splits["split_normalized"] == "test",
        "patient_id"
    ]
)

assert len(calibration_patient_ids) == 16
assert len(development_patient_ids) == 49
assert len(test_patient_ids) == 16

assert calibration_patient_ids.isdisjoint(
    development_patient_ids
)

assert calibration_patient_ids.isdisjoint(
    test_patient_ids
)

assert development_patient_ids.isdisjoint(
    test_patient_ids
)

print("\nPASS: Exactly 16 Calibration patients identified.")
print("PASS: Development and Calibration patients disjoint.")
print("PASS: Calibration and Test patients disjoint.")

# ------------------------------------------------------------
# 3. Load finalized 40x image metadata
# ------------------------------------------------------------

metadata_40x = pd.read_csv(
    METADATA_40X_PATH,
    dtype={"patient_id": str}
)

print("\n" + "=" * 70)
print("40x METADATA SCHEMA")
print("=" * 70)

print("Columns:")
print(metadata_40x.columns.tolist())

assert "patient_id" in metadata_40x.columns

# Select only Calibration records for subsequent work.
calibration_images = (
    metadata_40x.loc[
        metadata_40x["patient_id"].isin(
            calibration_patient_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Verify complete Calibration image membership
# ------------------------------------------------------------

assert len(calibration_images) == 382, (
    "Calibration image count differs from "
    f"the frozen protocol: {len(calibration_images)}"
)

assert (
    calibration_images["patient_id"].nunique()
    == 16
)

assert set(
    calibration_images["patient_id"]
) == calibration_patient_ids

# Every record must belong to a Calibration patient.
assert not calibration_images["patient_id"].isin(
    development_patient_ids | test_patient_ids
).any()

if "filename" in calibration_images.columns:

    assert calibration_images["filename"].is_unique

if "label" in calibration_images.columns:

    print("\nCalibration image label distribution:")
    print(
        calibration_images["label"]
        .value_counts()
        .sort_index()
        .to_string()
    )

print("\n" + "=" * 70)
print("FROZEN CALIBRATION COHORT")
print("=" * 70)

print("Calibration patients:", 16)
print("Calibration images:", len(calibration_images))

print("\nCalibration metadata preview:")
display(calibration_images.head(3))

# ------------------------------------------------------------
# 5. Verify against the original split assignments
# ------------------------------------------------------------

calibration_check = calibration_images[
    ["patient_id"]
].drop_duplicates().merge(
    patient_splits[
        ["patient_id", "split_normalized"]
    ],
    on="patient_id",
    how="left",
    validate="one_to_one"
)

assert len(calibration_check) == 16

assert calibration_check[
    "split_normalized"
].eq("calibration").all()

print("\n" + "=" * 70)
print("STAGE 7 — CALIBRATION COHORT AUDIT")
print("=" * 70)

print("PASS: Original frozen patient split loaded.")
print("PASS: Exactly 16 Calibration patients verified.")
print("PASS: Exactly 382 Calibration images verified.")
print("PASS: All patients match frozen assignments.")
print("PASS: No patient overlap with Development.")
print("PASS: No patient overlap with Test.")
print("PASS: No new partitions created.")
print("PASS: No CNN weights updated.")
print("PASS: No Calibration parameters fitted.")
print("PASS: No Test images accessed.")


FROZEN PATIENT PARTITIONS
split_normalized
development    49
calibration    16
test           16

PASS: Exactly 16 Calibration patients identified.
PASS: Development and Calibration patients disjoint.
PASS: Calibration and Test patients disjoint.

40x METADATA SCHEMA
Columns:
['filename', 'relative_path', 'patient_id', 'diagnosis_code', 'diagnosis', 'original_subtype_code', 'original_subtype', 'subtype_code', 'subtype', 'magnification', 'image_number', 'split']

FROZEN CALIBRATION COHORT
Calibration patients: 16
Calibration images: 382

Calibration metadata preview:


,filename,relative_path,patient_id,diagnosis_code,diagnosis,original_subtype_code,original_subtype,subtype_code,subtype,magnification,image_number,split
0,SOB_M_DC-14-11031-40-009.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,14-11031,M,Malignant,DC,Ductal carcinoma,DC,Ductal carcinoma,40,9,calibration
1,SOB_M_DC-14-11031-40-006.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,14-11031,M,Malignant,DC,Ductal carcinoma,DC,Ductal carcinoma,40,6,calibration
2,SOB_M_DC-14-11031-40-004.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,14-11031,M,Malignant,DC,Ductal carcinoma,DC,Ductal carcinoma,40,4,calibration



STAGE 7 — CALIBRATION COHORT AUDIT
PASS: Original frozen patient split loaded.
PASS: Exactly 16 Calibration patients verified.
PASS: Exactly 382 Calibration images verified.
PASS: All patients match frozen assignments.
PASS: No patient overlap with Development.
PASS: No patient overlap with Test.
PASS: No new partitions created.
PASS: No CNN weights updated.
PASS: No Calibration parameters fitted.
PASS: No Test images accessed.


In [ ]:

# ============================================================
# Load and Freeze the Final EfficientNet-B0 Model
# ============================================================

import hashlib
import zipfile
from google.colab import files
from torchvision.models import efficientnet_b0

# 1. Locate the saved final checkpoint

CHECKPOINT_PATH = Path(
    "/content/CIS631_model_checkpoints/"
    "efficientnet_b0/final_development/"
    "efficientnet_b0_final_epoch5.pt"
)

CHECKPOINT_PATH.parent.mkdir(
    parents=True, exist_ok=True
)

# 2. Upload only if the checkpoint is not available

if not CHECKPOINT_PATH.is_file():

    print("Final checkpoint not found in this Colab runtime.")
    print(
        "Upload CIS631_EfficientNetB0_Final_Stage6_Backup.zip "
        "or efficientnet_b0_final_epoch5.pt"
    )

    uploaded = files.upload()

    assert len(uploaded) == 1

    uploaded_path = Path(next(iter(uploaded)))

    if uploaded_path.suffix.lower() == ".zip":

        with zipfile.ZipFile(uploaded_path) as zf:

            matches = [
                name for name in zf.namelist()
                if Path(name).name ==
                "efficientnet_b0_final_epoch5.pt"
            ]

            assert len(matches) == 1, (
                "This ZIP does not contain the final "
                "model checkpoint. The GitHub metadata "
                "ZIP is not the correct backup."
            )

            with zf.open(matches[0]) as source:
                with open(CHECKPOINT_PATH, "wb") as target:
                    while True:
                        chunk = source.read(1024 * 1024)
                        if not chunk:
                            break
                        target.write(chunk)

    elif uploaded_path.name == "efficientnet_b0_final_epoch5.pt":

        CHECKPOINT_PATH.write_bytes(
            uploaded_path.read_bytes()
        )

    else:
        raise ValueError("Incorrect checkpoint file uploaded.")

else:
    print("Existing final checkpoint found. No upload needed.")

# 3. Verify SHA-256

digest = hashlib.sha256()

with open(CHECKPOINT_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        digest.update(chunk)

assert digest.hexdigest() == EXPECTED_CHECKPOINT_SHA256, (
    "Checkpoint SHA-256 does not match Stage 6."
)

print("PASS: Final checkpoint SHA-256 verified.")

# 4. Reconstruct the model

calibration_model = efficientnet_b0(weights=None)

calibration_model.classifier[-1] = torch.nn.Linear(
    1280, 1
)

# 5. Load final trained weights

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

assert checkpoint["epoch_completed"] == 5
assert checkpoint["architecture"] == "efficientnet_b0"

calibration_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

# 6. Freeze model parameters

calibration_model = calibration_model.to(DEVICE)
calibration_model.eval()

for parameter in calibration_model.parameters():
    parameter.requires_grad_(False)

assert all(
    not p.requires_grad
    for p in calibration_model.parameters()
)

# 7. Verify model inference

with torch.inference_mode():
    logits = calibration_model(
        torch.zeros(1, 3, 224, 224, device=DEVICE)
    )

assert logits.shape == (1, 1)
assert torch.isfinite(logits).all()

del checkpoint, logits

print("\n" + "=" * 60)
print("STAGE 7 — MODEL RESTORATION")
print("=" * 60)

print("PASS: Final Epoch 5 checkpoint loaded.")
print("PASS: Original model weights verified.")
print("PASS: All CNN parameters frozen.")
print("PASS: Inference mode enabled.")
print("PASS: Forward-pass verification successful.")
print("PASS: No model training performed.")
print("PASS: No Test data accessed.")


Final checkpoint not found in this Colab runtime.
Upload CIS631_EfficientNetB0_Final_Stage6_Backup.zip or efficientnet_b0_final_epoch5.pt


Saving CIS631_EfficientNetB0_Final_Stage6_Backup.zip to CIS631_EfficientNetB0_Final_Stage6_Backup.zip
PASS: Final checkpoint SHA-256 verified.

STAGE 7 — MODEL RESTORATION
PASS: Final Epoch 5 checkpoint loaded.
PASS: Original model weights verified.
PASS: All CNN parameters frozen.
PASS: Inference mode enabled.
PASS: Forward-pass verification successful.
PASS: No model training performed.
PASS: No Test data accessed.


In [ ]:

# ============================================================
# Audit Original Development Patch Extraction
# ============================================================

import ast

SOURCE_NOTEBOOK_PATH = (
    REPO_ROOT /
    "Notebooks/03_breakhis_formal_development.ipynb"
)

DEVELOPMENT_PATCH_MANIFEST_PATH = (
    REPO_ROOT /
    "Metadata/breakhis_development_all_patch_manifest.csv"
)

assert SOURCE_NOTEBOOK_PATH.is_file()
assert DEVELOPMENT_PATCH_MANIFEST_PATH.is_file()

# 1. Inspect the frozen Development patch manifest.

development_patch_manifest = pd.read_csv(
    DEVELOPMENT_PATCH_MANIFEST_PATH
)

assert len(development_patch_manifest) == 17895
assert development_patch_manifest["patient_id"].nunique() == 49
assert development_patch_manifest["filename"].nunique() == 1133

print("=" * 70)
print("ORIGINAL DEVELOPMENT PATCH MANIFEST")
print("=" * 70)

print("Patches:", len(development_patch_manifest))
print("Images:", development_patch_manifest["filename"].nunique())
print("Patients:", development_patch_manifest["patient_id"].nunique())

print("\nManifest columns:")
print(development_patch_manifest.columns.tolist())

print("\nFirst two patch records:")
display(development_patch_manifest.head(2))

patch_counts = (
    development_patch_manifest
    .groupby("filename")
    .size()
)

print("\nPatches per source image:")
print(patch_counts.describe().to_string())

# 2. Read the original preprocessing notebook.

with open(
    SOURCE_NOTEBOOK_PATH,
    encoding="utf-8"
) as f:
    source_notebook = json.load(f)

print("\n" + "=" * 70)
print("ORIGINAL PREPROCESSING DEFINITIONS")
print("=" * 70)

relevant_terms = (
    "patch",
    "tile",
    "tissue",
    "resample",
    "manifest"
)

relevant_definitions = []

for cell_index, cell in enumerate(
    source_notebook["cells"]
):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:

        if not isinstance(
            node,
            (ast.FunctionDef, ast.ClassDef)
        ):
            continue

        if any(
            term in node.name.lower()
            for term in relevant_terms
        ):

            relevant_definitions.append(
                (
                    cell_index,
                    node.name,
                    ast.get_source_segment(source, node)
                )
            )

for cell_index, name, source in relevant_definitions:

    print(f"\nNotebook cell {cell_index}: {name}")

    # Show the original function/class implementation.
    # Larger definitions can be examined separately.
    if source is not None:
        print(source[:9000])

        if len(source) > 9000:
            print("[Definition truncated after 9000 characters]")

# 3. Find original configuration and patch generation cells.

print("\n" + "=" * 70)
print("PATCH CONFIGURATION / GENERATION CELLS")
print("=" * 70)

search_terms = (
    "SOURCE_MPP_BREAKHIS",
    "TARGET_MPP",
    "PATCH_SIZE",
    "PATCH_STRIDE",
    "patch_index",
    "tissue_fraction",
    "patch_manifest"
)

candidate_cells = []

for cell_index, cell in enumerate(
    source_notebook["cells"]
):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    matching_terms = [
        term for term in search_terms
        if term in source
    ]

    if matching_terms:

        candidate_cells.append(
            (cell_index, matching_terms, source)
        )

for cell_index, matching_terms, source in candidate_cells:

    print(
        f"Cell {cell_index}: "
        + ", ".join(matching_terms)
    )

# 4. Confirm the Calibration cohort remains isolated.

assert len(calibration_images) == 382
assert calibration_images["patient_id"].nunique() == 16

assert set(
    calibration_images["patient_id"]
).isdisjoint(
    set(development_patch_manifest["patient_id"])
)

print("\n" + "=" * 70)
print("STAGE 7 — PATCH PROTOCOL PREFLIGHT")
print("=" * 70)

print("PASS: Original preprocessing notebook located.")
print("PASS: Original Development manifest verified.")
print("PASS: Patch extraction definitions identified.")
print("PASS: Frozen Calibration cohort still contains 382 images.")
print("PASS: Calibration and Development patients disjoint.")
print("PASS: No image inference performed.")
print("PASS: No CNN weights modified.")
print("PASS: No Test data accessed.")


ORIGINAL DEVELOPMENT PATCH MANIFEST
Patches: 17895
Images: 1133
Patients: 49

Manifest columns:
['patient_id', 'filename', 'relative_path', 'diagnosis', 'label', 'patch_index', 'x', 'y', 'tissue_fraction', 'used_fallback', 'development_fold']

First two patch records:


,patient_id,filename,relative_path,diagnosis,label,patch_index,x,y,tissue_fraction,used_fallback,development_fold
0,14-13200,SOB_B_TA-14-13200-40-001.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,Benign,0,0,0,0,0.839465,False,1
1,14-13200,SOB_B_TA-14-13200-40-001.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,Benign,0,1,154,0,0.592056,False,1



Patches per source image:
count    1133.000000
mean       15.794351
std         1.121223
min         1.000000
25%        16.000000
50%        16.000000
75%        16.000000
max        16.000000

ORIGINAL PREPROCESSING DEFINITIONS

Notebook cell 19: resample_to_target_mpp
def resample_to_target_mpp(
    image,
    source_mpp=SOURCE_MPP_BREAKHIS,
    target_mpp=TARGET_MPP
):
    """
    Resample the source image to 0.50 micrometers/pixel.
    Preserve the original Notebook 01 implementation.
    """

    image = image.convert("RGB")

    scale = source_mpp / target_mpp

    new_width = round(image.width * scale)
    new_height = round(image.height * scale)

    return image.resize(
        (new_width, new_height),
        Image.Resampling.LANCZOS
    )

Notebook cell 20: BreakHisPatchDataset
class BreakHisPatchDataset(Dataset):

    def __init__(
        self,
        manifest,
        dataset_root,
        transform=None
    ):

        self.manifest = (
            manifest
          

In [ ]:

# ============================================================
# Locate Original Patch Generation and Tissue Filtering
# ============================================================

NOTEBOOK_DIR = REPO_ROOT / "Notebooks"

original_notebooks = sorted(
    path
    for path in NOTEBOOK_DIR.glob("*.ipynb")
    if path.name[:2] in {"01", "02", "03"}
)

assert original_notebooks

print("=" * 70)
print("ORIGINAL NOTEBOOK VALIDATION")
print("=" * 70)

valid_notebooks = []
invalid_notebooks = []

for notebook_path in original_notebooks:

    try:
        with open(
            notebook_path,
            encoding="utf-8"
        ) as f:
            notebook = json.load(f)

        assert isinstance(notebook, dict)
        assert isinstance(notebook.get("cells"), list)

        valid_notebooks.append(
            (notebook_path, notebook)
        )

        print("VALID:", notebook_path.name)

    except (
        json.JSONDecodeError,
        UnicodeDecodeError,
        AssertionError
    ) as error:

        invalid_notebooks.append(
            (notebook_path.name, str(error))
        )

        print("SKIPPED:", notebook_path.name)
        print(" Reason:", error)

assert valid_notebooks, (
    "No valid original notebooks were found."
)

# The original Stage 3 notebook was successfully read
# in Cell 5, so it must also be valid here.

assert any(
    path.name == "03_breakhis_formal_development.ipynb"
    for path, _ in valid_notebooks
)

print("\nValid notebooks:", len(valid_notebooks))
print("Skipped notebooks:", len(invalid_notebooks))

# ------------------------------------------------------------
# Search only valid notebook CODE cells
# ------------------------------------------------------------

search_terms = (
    "tissue_fraction",
    "used_fallback",
    "tissue_threshold",
    "tissue_mask",
    "patch_coordinates",
    "patch_positions",
    "PATCH_SIZE",
    "PATCH_STRIDE",
    "MAX_PATCHES",
    "patch_index"
)

matches = []

for notebook_path, notebook in valid_notebooks:

    for cell_number, cell in enumerate(
        notebook["cells"]
    ):

        if cell.get("cell_type") != "code":
            continue

        source = "".join(
            cell.get("source", [])
        )

        found = [
            term
            for term in search_terms
            if term.lower() in source.lower()
        ]

        if not found:
            continue

        score = len(found)

        if "tissue_fraction" in source:
            score += 5

        if "used_fallback" in source:
            score += 7

        if (
            "tissue_fraction" in source
            and "used_fallback" in source
        ):
            score += 10

        if (
            "def " in source
            and "patch" in source.lower()
        ):
            score += 5

        if (
            "patch_index" in source
            and ".append(" in source
        ):
            score += 5

        if (
            "Image.open(" in source
            and "patch" in source.lower()
        ):
            score += 3

        matches.append({
            "notebook": notebook_path.name,
            "cell": cell_number,
            "score": score,
            "terms": found,
            "source": source
        })

assert matches, (
    "No relevant patch-processing code was found "
    "in the valid notebooks."
)

matches.sort(
    key=lambda item: (
        -item["score"],
        item["notebook"],
        item["cell"]
    )
)

print("\n" + "=" * 70)
print("PATCH GENERATION SOURCE CANDIDATES")
print("=" * 70)

for item in matches[:12]:

    print(
        f"{item['notebook']} | "
        f"Cell {item['cell']} | "
        f"Score {item['score']} | "
        f"Matches: {', '.join(item['terms'])}"
    )

print("\n" + "=" * 70)
print("ORIGINAL CODE — TOP FOUR MATCHES")
print("=" * 70)

for item in matches[:4]:

    print("\n" + "-" * 70)
    print(
        f"FILE: {item['notebook']}\n"
        f"CELL: {item['cell']}"
    )
    print("-" * 70)

    print(item["source"][:12000])

    if len(item["source"]) > 12000:
        print("[Source truncated]")

# ------------------------------------------------------------
# Verify that the frozen model is still untouched
# ------------------------------------------------------------

assert len(calibration_images) == 382
assert calibration_images["patient_id"].nunique() == 16

assert not calibration_model.training

assert all(
    not parameter.requires_grad
    for parameter in calibration_model.parameters()
)

print("\n" + "=" * 70)
print("STAGE 7 — PATCH SOURCE AUDIT")
print("=" * 70)

print("PASS: Valid original notebooks inspected.")
print("PASS: Unreadable notebooks handled safely.")
print("PASS: Patch-processing candidates identified.")
print("PASS: Calibration cohort remains unchanged.")
print("PASS: Final CNN remains frozen.")
print("PASS: No CNN training performed.")
print("PASS: No Test data accessed.")


ORIGINAL NOTEBOOK VALIDATION
SKIPPED: 01_breakhis_audit.ipynb
 Reason: Expecting value: line 1 column 1 (char 0)
VALID: 01_breakhis_data_audit_and_split.ipynb
VALID: 02_breakhis_all_magnification_audit.ipynb
SKIPPED: 02_preliminary_baseline.ipynb
 Reason: Expecting value: line 1 column 1 (char 0)
VALID: 03_breakhis_formal_development.ipynb

Valid notebooks: 3
Skipped notebooks: 2

PATCH GENERATION SOURCE CANDIDATES
01_breakhis_data_audit_and_split.ipynb | Cell 66 | Score 38 | Matches: tissue_fraction, used_fallback, tissue_mask, PATCH_SIZE, MAX_PATCHES, patch_index
01_breakhis_data_audit_and_split.ipynb | Cell 84 | Score 38 | Matches: tissue_fraction, used_fallback, patch_index
01_breakhis_data_audit_and_split.ipynb | Cell 70 | Score 27 | Matches: tissue_fraction, used_fallback
03_breakhis_formal_development.ipynb | Cell 18 | Score 17 | Matches: used_fallback, patch_index
01_breakhis_data_audit_and_split.ipynb | Cell 67 | Score 11 | Matches: used_fallback
03_breakhis_formal_development

In [ ]:

# ============================================================
# Verify Original Tissue Mask, Grid, and Patch Constants
# ============================================================

SOURCE_PATH = (
    REPO_ROOT /
    "Notebooks/01_breakhis_data_audit_and_split.ipynb"
)

with open(SOURCE_PATH, encoding="utf-8") as f:
    source_nb = json.load(f)

# Index original function definitions and constant values.

function_sources = {}
constant_sources = {}

required_functions = {
    "resample_to_target_mpp",
    "create_tissue_mask_hybrid",
    "generate_grid_coordinates",
}

important_constants = {
    "SOURCE_MPP_BREAKHIS",
    "TARGET_MPP",
    "PATCH_SIZE",
    "MAX_PATCHES",
    "MIN_PATCH_TISSUE_FRACTION",
    "OD_THRESHOLD",
    "SATURATION_THRESHOLD",
    "PATCH_STRIDE",
    "STRIDE",
}

for cell_number, cell in enumerate(source_nb["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:

        if isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef)
        ):

            function_sources[node.name] = {
                "cell": cell_number,
                "source": ast.get_source_segment(
                    source, node
                ),
                "node": node
            }

        if isinstance(node, ast.Assign):
            target_names = [
                t.id
                for t in node.targets
                if isinstance(t, ast.Name)
            ]

        elif isinstance(node, ast.AnnAssign):
            target_names = (
                [node.target.id]
                if isinstance(node.target, ast.Name)
                else []
            )

        else:
            target_names = []

        for name in target_names:

            if name in important_constants:
                constant_sources[name] = {
                    "cell": cell_number,
                    "source": ast.get_source_segment(
                        source, node
                    )
                }

# ------------------------------------------------------------
# 1. Original preprocessing constants
# ------------------------------------------------------------

print("=" * 70)
print("ORIGINAL PREPROCESSING CONSTANTS")
print("=" * 70)

for name in sorted(important_constants):

    if name in constant_sources:

        record = constant_sources[name]

        print(
            f"Cell {record['cell']}: "
            f"{record['source']}"
        )

    else:
        print(f"NOT EXPLICITLY DEFINED: {name}")

# ------------------------------------------------------------
# 2. Find all helper functions used by required functions
# ------------------------------------------------------------

missing_required = (
    required_functions - set(function_sources)
)

assert not missing_required, (
    "Missing original helper definitions: "
    f"{sorted(missing_required)}"
)

functions_to_show = set()
pending = list(required_functions)

while pending:

    name = pending.pop()

    if name in functions_to_show:
        continue

    functions_to_show.add(name)

    node = function_sources[name]["node"]

    for child in ast.walk(node):

        if isinstance(child, ast.Name):

            dependency = child.id

            if (
                dependency in function_sources
                and dependency not in functions_to_show
            ):
                pending.append(dependency)

# ------------------------------------------------------------
# 3. Print original definitions without modifying them
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ORIGINAL PREPROCESSING HELPERS")
print("=" * 70)

for name in sorted(functions_to_show):

    record = function_sources[name]

    print("\n" + "-" * 70)
    print(f"{name} — Original Cell {record['cell']}")
    print("-" * 70)

    print(record["source"])

# ------------------------------------------------------------
# 4. Confirm model and Calibration isolation
# ------------------------------------------------------------

assert len(calibration_images) == 382

assert (
    calibration_images["patient_id"].nunique()
    == 16
)

assert not calibration_model.training

assert all(
    not parameter.requires_grad
    for parameter in calibration_model.parameters()
)

print("\n" + "=" * 70)
print("STAGE 7 — PREPROCESSING VERIFICATION")
print("=" * 70)

print("PASS: Original patch-generation notebook loaded.")
print("PASS: Original constants inspected.")
print("PASS: Required preprocessing helpers found.")
print("PASS: Helper dependencies inspected.")
print("PASS: Calibration cohort unchanged.")
print("PASS: Final CNN remains frozen.")
print("PASS: No image processing performed.")
print("PASS: No Test data accessed.")


ORIGINAL PREPROCESSING CONSTANTS
Cell 63: MAX_PATCHES = 16
Cell 63: MIN_PATCH_TISSUE_FRACTION = 0.20
Cell 63: OD_THRESHOLD = 0.08
Cell 97: PATCH_SIZE = 224
NOT EXPLICITLY DEFINED: PATCH_STRIDE
Cell 63: SATURATION_THRESHOLD = 0.05
Cell 97: SOURCE_MPP_BREAKHIS = 0.49
NOT EXPLICITLY DEFINED: STRIDE
Cell 97: TARGET_MPP = 0.50

ORIGINAL PREPROCESSING HELPERS

----------------------------------------------------------------------
create_tissue_mask_hybrid — Original Cell 53
----------------------------------------------------------------------
def create_tissue_mask_hybrid(
    image,
    od_threshold=OD_THRESHOLD,
    saturation_threshold=SATURATION_THRESHOLD
):
    """
    Development-only prototype tissue mask.

    A pixel is considered tissue when it has:
    1. sufficient optical density, AND
    2. sufficient color saturation.
    """

    rgb = np.asarray(
        image.convert("RGB"),
        dtype=np.float32
    ) / 255.0

    rgb = np.clip(rgb, 1 / 255.0, 1.0)

    # Optical densi

In [ ]:

# ============================================================
# Reproduce Preprocessing, Verify Calibration Manifest,
# and Save Complete Audit
# ============================================================

# Only libraries newly needed in this stage.
import numpy as np
import kagglehub
from PIL import Image
from matplotlib.colors import rgb_to_hsv

# ------------------------------------------------------------
# 1. SHA-256 helper (included before first use)
# ------------------------------------------------------------

def stage7_sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 2. Verify and restore original preprocessing constants
# ------------------------------------------------------------

locked_constants = {
    "SOURCE_MPP_BREAKHIS": 0.49,
    "TARGET_MPP": 0.50,
    "PATCH_SIZE": 224,
    "GRID_SIZE": 4,
    "MAX_PATCHES": 16,
    "MIN_PATCH_TISSUE_FRACTION": 0.20,
    "OD_THRESHOLD": 0.08,
    "SATURATION_THRESHOLD": 0.05
}

for name, expected in locked_constants.items():

    original_values = []

    for cell in source_nb["cells"]:

        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue

        for node in tree.body:

            if isinstance(node, ast.Assign):
                targets = [
                    target.id
                    for target in node.targets
                    if isinstance(target, ast.Name)
                ]
                value_node = node.value

            elif isinstance(node, ast.AnnAssign):
                targets = (
                    [node.target.id]
                    if isinstance(node.target, ast.Name)
                    else []
                )
                value_node = node.value

            else:
                continue

            if name in targets and value_node is not None:

                try:
                    original_values.append(
                        ast.literal_eval(value_node)
                    )
                except (ValueError, TypeError):
                    pass

    assert original_values, (
        f"Original constant missing: {name}"
    )

    actual = original_values[-1]

    assert actual == expected, (
        f"{name}: original={actual}, expected={expected}"
    )

    globals()[name] = actual

print("PASS: All eight original preprocessing constants verified.")

# ------------------------------------------------------------
# 3. Restore original preprocessing functions
# ------------------------------------------------------------

required_helpers = [
    "resample_to_target_mpp",
    "create_tissue_mask_hybrid",
    "generate_grid_coordinates",
    "extract_tissue_patches",
    "build_patch_manifest"
]

for name in required_helpers:

    assert name in function_sources

    source = function_sources[name]["source"]

    assert source is not None

    exec(source, globals())

assert len(
    generate_grid_coordinates(686, 451)
) == 16

print("PASS: Five original preprocessing functions restored.")
print("PASS: Original 4 x 4 grid verified.")

# ------------------------------------------------------------
# 4. Locate original BreaKHis dataset
# ------------------------------------------------------------

DATASET_ROOT = Path("/kaggle/input/breakhis")

if not DATASET_ROOT.is_dir():

    DATASET_ROOT = Path(
        kagglehub.dataset_download(
            "ambarish/breakhis"
        )
    )

assert DATASET_ROOT.is_dir()

# Original manifest builder uses the variable 'root'.
root = DATASET_ROOT

calibration_paths = [
    DATASET_ROOT / relative_path
    for relative_path in calibration_images["relative_path"]
]

assert len(calibration_paths) == 382

assert all(
    path.is_file()
    for path in calibration_paths
), "Some Calibration source images are missing."

print("PASS: All 382 Calibration source images located.")

# ------------------------------------------------------------
# 5. Reproduce original Development extraction
# ------------------------------------------------------------

sample_filenames = []

for label in [0, 1]:

    sample_filenames.append(
        development_patch_manifest.loc[
            development_patch_manifest["label"] == label,
            "filename"
        ].iloc[0]
    )

fallback_rows = development_patch_manifest.loc[
    development_patch_manifest["used_fallback"]
    .astype(str)
    .str.lower()
    .eq("true")
]

if not fallback_rows.empty:

    sample_filenames.append(
        fallback_rows.iloc[0]["filename"]
    )

sample_filenames = list(
    dict.fromkeys(sample_filenames)
)

for filename in sample_filenames:

    expected = (
        development_patch_manifest.loc[
            development_patch_manifest["filename"] == filename
        ]
        .sort_values("patch_index")
        .reset_index(drop=True)
    )

    image_path = (
        DATASET_ROOT /
        expected.iloc[0]["relative_path"]
    )

    assert image_path.is_file()

    with Image.open(image_path) as image:

        _, patch_info, used_fallback = (
            extract_tissue_patches(image)
        )

    actual = (
        pd.DataFrame(patch_info)
        .sort_values("patch_index")
        .reset_index(drop=True)
    )

    assert len(actual) == len(expected)

    for column in ["patch_index", "x", "y"]:

        assert np.array_equal(
            actual[column].to_numpy(),
            expected[column].to_numpy()
        ), (
            f"Coordinate mismatch for {filename}"
        )

    np.testing.assert_allclose(
        actual["tissue_fraction"].to_numpy(),
        expected["tissue_fraction"].to_numpy(),
        atol=1e-7,
        rtol=0
    )

    expected_fallback = (
        expected["used_fallback"]
        .astype(str)
        .str.lower()
        .eq("true")
        .all()
    )

    assert used_fallback == expected_fallback

    print(
        f"PASS: Development extraction reproduced: "
        f"{filename}"
    )

# ------------------------------------------------------------
# 6. Load existing Calibration manifest or generate once
# ------------------------------------------------------------

CALIBRATION_RESULTS_DIR = (
    REPO_ROOT /
    "results/posthoc_calibration/efficientnet_b0"
)

CALIBRATION_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CALIBRATION_MANIFEST_PATH = (
    REPO_ROOT /
    "Metadata/breakhis_calibration_40x_patch_manifest.csv"
)

if CALIBRATION_MANIFEST_PATH.is_file():

    # Original Cell 8 already saved this manifest.
    # Reuse it instead of processing 382 images again.

    calibration_patch_manifest = pd.read_csv(
        CALIBRATION_MANIFEST_PATH,
        dtype={"patient_id": str}
    )

    print("\nExisting Calibration manifest found.")
    print("Reusing saved patches; no regeneration needed.")

else:

    print("\nGenerating Calibration patch manifest...")

    calibration_patch_manifest = build_patch_manifest(
        calibration_images
    )

    temporary_path = (
        CALIBRATION_MANIFEST_PATH.with_suffix(
            ".csv.tmp"
        )
    )

    calibration_patch_manifest.to_csv(
        temporary_path,
        index=False
    )

    temporary_path.replace(
        CALIBRATION_MANIFEST_PATH
    )

    print("New Calibration manifest generated and saved.")

# ------------------------------------------------------------
# 7. Validate Calibration manifest
# ------------------------------------------------------------

required_columns = {
    "patient_id",
    "filename",
    "relative_path",
    "diagnosis",
    "label",
    "patch_index",
    "x",
    "y",
    "tissue_fraction",
    "used_fallback"
}

assert required_columns.issubset(
    calibration_patch_manifest.columns
)

assert len(calibration_patch_manifest) == 6079

assert (
    calibration_patch_manifest["filename"].nunique()
    == 382
)

assert (
    calibration_patch_manifest["patient_id"].nunique()
    == 16
)

assert set(
    calibration_patch_manifest["patient_id"]
) == calibration_patient_ids

assert set(
    calibration_patch_manifest["filename"]
) == set(calibration_images["filename"])

assert not calibration_patch_manifest.duplicated(
    ["filename", "patch_index"]
).any()

assert calibration_patch_manifest[
    "patch_index"
].between(0, 15).all()

assert calibration_patch_manifest[
    "tissue_fraction"
].between(0, 1).all()

assert set(
    calibration_patch_manifest["label"].unique()
) == {0, 1}

assert set(
    calibration_patch_manifest["filename"]
).isdisjoint(
    set(development_patch_manifest["filename"])
)

# Check every patch belongs to the correct frozen patient,
# path, and diagnosis.

image_reference = calibration_images.set_index(
    "filename"
)

assert image_reference.index.is_unique

for column in [
    "patient_id",
    "relative_path",
    "diagnosis"
]:

    expected_values = (
        calibration_patch_manifest["filename"]
        .map(image_reference[column])
        .astype(str)
        .to_numpy()
    )

    actual_values = (
        calibration_patch_manifest[column]
        .astype(str)
        .to_numpy()
    )

    assert np.array_equal(
        actual_values,
        expected_values
    ), f"Calibration {column} mismatch."

expected_labels = (
    calibration_patch_manifest["diagnosis"]
    .eq("Malignant")
    .astype(int)
    .to_numpy()
)

assert np.array_equal(
    calibration_patch_manifest["label"].to_numpy(),
    expected_labels
)

patches_per_image = (
    calibration_patch_manifest
    .groupby("filename")
    .size()
)

assert patches_per_image.between(1, 16).all()

fallback_images = (
    calibration_patch_manifest.loc[
        calibration_patch_manifest["used_fallback"]
        .astype(str)
        .str.lower()
        .eq("true"),
        "filename"
    ]
    .nunique()
)

assert fallback_images == 0

image_labels = (
    calibration_patch_manifest
    .drop_duplicates("filename")["label"]
    .value_counts()
    .to_dict()
)

assert image_labels.get(0, 0) == 121
assert image_labels.get(1, 0) == 261

# Confirm the final trained CNN is still frozen.

assert not calibration_model.training

assert all(
    not parameter.requires_grad
    for parameter in calibration_model.parameters()
)

print("\n" + "=" * 65)
print("CALIBRATION PATCH MANIFEST SUMMARY")
print("=" * 65)

print("Patients:", 16)
print("Images:", 382)
print("Patches:", len(calibration_patch_manifest))
print("Benign images:", 121)
print("Malignant images:", 261)
print("Fallback images:", fallback_images)

print("\nPatches per image:")
print(patches_per_image.describe().to_string())

# ------------------------------------------------------------
# 8. Save the complete preprocessing audit
# ------------------------------------------------------------

audit_record = {
    "stage": 7,
    "cohort": "Calibration",
    "magnification": "40x",
    "patients": 16,
    "images": 382,
    "patches": int(len(calibration_patch_manifest)),
    "benign_images": 121,
    "malignant_images": 261,
    "fallback_images": int(fallback_images),
    "patch_size": PATCH_SIZE,
    "grid_size": GRID_SIZE,
    "max_patches": MAX_PATCHES,
    "min_tissue_fraction":
        MIN_PATCH_TISSUE_FRACTION,
    "od_threshold": OD_THRESHOLD,
    "saturation_threshold": SATURATION_THRESHOLD,
    "source_mpp": SOURCE_MPP_BREAKHIS,
    "target_mpp": TARGET_MPP,
    "original_preprocessing_notebook":
        SOURCE_PATH.name,
    "original_preprocessing_notebook_sha256":
        stage7_sha256(SOURCE_PATH),
    "calibration_manifest_sha256":
        stage7_sha256(CALIBRATION_MANIFEST_PATH),
    "final_model_sha256":
        EXPECTED_CHECKPOINT_SHA256,
    "development_reproduction_verified": True,
    "test_data_accessed": False,
    "cnn_weights_modified": False,
    "calibrators_fitted": False
}

CALIBRATION_AUDIT_PATH = (
    CALIBRATION_RESULTS_DIR /
    "calibration_patch_manifest_audit.json"
)

with open(
    CALIBRATION_AUDIT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit_record,
        f,
        indent=2
    )

assert CALIBRATION_AUDIT_PATH.is_file()

with open(
    CALIBRATION_AUDIT_PATH,
    encoding="utf-8"
) as f:

    verified_audit = json.load(f)

assert verified_audit["patches"] == 6079

assert verified_audit[
    "calibration_manifest_sha256"
] == stage7_sha256(CALIBRATION_MANIFEST_PATH)

# ------------------------------------------------------------
# 9. Final audit
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STAGE 7 — CALIBRATION PATCH AUDIT COMPLETE")
print("=" * 65)

print("PASS: Original preprocessing constants verified.")
print("PASS: Original extraction functions restored.")
print("PASS: Development extraction reproduced.")
print("PASS: Exactly 16 Calibration patients.")
print("PASS: Exactly 382 Calibration images.")
print("PASS: Exactly 6,079 Calibration patches.")
print("PASS: Calibration manifest saved and verified.")
print("PASS: Manifest SHA-256 verified.")
print("PASS: Complete preprocessing audit JSON saved.")
print("PASS: Final CNN remains frozen.")
print("PASS: No Test data accessed.")
print("PASS: No calibration parameters fitted.")

print("\nManifest:", CALIBRATION_MANIFEST_PATH)

print(
    "Manifest SHA-256:",
    verified_audit["calibration_manifest_sha256"]
)

print("Audit:", CALIBRATION_AUDIT_PATH)


PASS: All eight original preprocessing constants verified.
PASS: Five original preprocessing functions restored.
PASS: Original 4 x 4 grid verified.
PASS: All 382 Calibration source images located.
PASS: Development extraction reproduced: SOB_B_TA-14-13200-40-001.png
PASS: Development extraction reproduced: SOB_M_MC-14-13418DE-40-001.png

Existing Calibration manifest found.
Reusing saved patches; no regeneration needed.

CALIBRATION PATCH MANIFEST SUMMARY
Patients: 16
Images: 382
Patches: 6079
Benign images: 121
Malignant images: 261
Fallback images: 0

Patches per image:
count    382.000000
mean      15.913613
std        0.606503
min        9.000000
25%       16.000000
50%       16.000000
75%       16.000000
max       16.000000

STAGE 7 — CALIBRATION PATCH AUDIT COMPLETE
PASS: Original preprocessing constants verified.
PASS: Original extraction functions restored.
PASS: Development extraction reproduced.
PASS: Exactly 16 Calibration patients.
PASS: Exactly 382 Calibration images.
PAS

In [ ]:

# ============================================================
# Frozen EfficientNet-B0 Calibration Inference
# ============================================================

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. Verify Calibration inputs and frozen model
# ------------------------------------------------------------

assert len(calibration_patch_manifest) == 6079
assert calibration_patch_manifest["filename"].nunique() == 382
assert calibration_patch_manifest["patient_id"].nunique() == 16

assert stage7_sha256(
    CALIBRATION_MANIFEST_PATH
) == (
    "249bc344b2e6a286c6d0968b292c36b479a1325e83752fcc2d0493bbfe41cce5"
)

assert stage7_sha256(
    CHECKPOINT_PATH
) == EXPECTED_CHECKPOINT_SHA256

assert not calibration_model.training

assert all(
    not parameter.requires_grad
    for parameter in calibration_model.parameters()
)

# Preserve all model tensors for an independent
# before/after check that inference changes nothing.

model_state_before = {
    name: tensor.detach().cpu().clone()
    for name, tensor in calibration_model.state_dict().items()
}

print("PASS: Calibration manifest verified.")
print("PASS: Final Epoch 5 model identity verified.")
print("PASS: All CNN parameters frozen.")

# ------------------------------------------------------------
# 2. Restore original Development patch dataset class
# ------------------------------------------------------------

with open(
    SOURCE_NOTEBOOK_PATH,
    encoding="utf-8"
) as f:
    development_notebook = json.load(f)

dataset_definitions = []

for cell in development_notebook["cells"]:

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:

        if (
            isinstance(node, ast.ClassDef)
            and node.name == "BreakHisPatchDataset"
        ):

            dataset_definitions.append(
                ast.get_source_segment(source, node)
            )

assert dataset_definitions

assert all(
    definition is not None
    for definition in dataset_definitions
)

# Restore exactly the original class implementation.
exec(dataset_definitions[-1], globals())

print("PASS: Original BreakHisPatchDataset restored.")

# ------------------------------------------------------------
# 3. Define deterministic inference preprocessing
# ------------------------------------------------------------

# Consistent with ImageNet-normalized Development
# inference; no random augmentation.

calibration_inference_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

calibration_dataset = BreakHisPatchDataset(
    manifest=calibration_patch_manifest.copy(),
    dataset_root=DATASET_ROOT,
    transform=calibration_inference_transform
)

assert len(calibration_dataset) == 6079

CALIBRATION_BATCH_SIZE = 32

calibration_loader = DataLoader(
    calibration_dataset,
    batch_size=CALIBRATION_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

assert len(calibration_loader) == 190

print("PASS: Deterministic inference transform prepared.")
print("PASS: 6,079 patches loaded into inference dataset.")
print("PASS: DataLoader configured without shuffling.")

# ------------------------------------------------------------
# 4. Perform frozen CNN inference
# ------------------------------------------------------------

calibration_model.eval()

all_patch_logits = []
all_filenames = []
all_patient_ids = []
all_patch_indices = []
all_labels = []

print("\n" + "=" * 70)
print("CALIBRATION PATCH INFERENCE")
print("=" * 70)

with torch.inference_mode():

    for batch in tqdm(
        calibration_loader,
        desc="Frozen EfficientNet-B0 inference"
    ):

        images = batch["image"].to(
            DEVICE,
            non_blocking=True
        )

        logits = calibration_model(
            images
        ).reshape(-1)

        assert torch.isfinite(logits).all()

        all_patch_logits.extend(
            logits.float().cpu().tolist()
        )

        all_filenames.extend(
            list(batch["filename"])
        )

        all_patient_ids.extend(
            list(batch["patient_id"])
        )

        all_patch_indices.extend(
            batch["patch_index"].cpu().tolist()
        )

        all_labels.extend(
            batch["label"].cpu().tolist()
        )

assert len(all_patch_logits) == 6079

assert np.isfinite(
    np.asarray(all_patch_logits)
).all()

print("\nPASS: Inference completed for 6,079 patches.")

# ------------------------------------------------------------
# 5. Verify prediction-to-manifest alignment
# ------------------------------------------------------------

expected_manifest = (
    calibration_patch_manifest
    .reset_index(drop=True)
)

assert all_filenames == (
    expected_manifest["filename"]
    .astype(str)
    .tolist()
)

assert all_patient_ids == (
    expected_manifest["patient_id"]
    .astype(str)
    .tolist()
)

assert all_patch_indices == (
    expected_manifest["patch_index"]
    .astype(int)
    .tolist()
)

np.testing.assert_array_equal(
    np.asarray(all_labels, dtype=int),
    expected_manifest["label"].to_numpy(dtype=int)
)

print("PASS: Every prediction matches its original patch.")
print("PASS: Patient, filename, patch index and label aligned.")

# ------------------------------------------------------------
# 6. Construct patch-level prediction table
# ------------------------------------------------------------

calibration_patch_predictions = (
    expected_manifest[
        [
            "patient_id",
            "filename",
            "relative_path",
            "label",
            "patch_index",
            "x",
            "y",
            "tissue_fraction",
            "used_fallback"
        ]
    ]
    .copy()
)

calibration_patch_predictions["patch_logit"] = (
    np.asarray(all_patch_logits, dtype=np.float64)
)

calibration_patch_predictions["patch_probability"] = (
    torch.sigmoid(
        torch.tensor(
            all_patch_logits,
            dtype=torch.float64
        )
    )
    .numpy()
)

assert not calibration_patch_predictions.duplicated(
    ["filename", "patch_index"]
).any()

assert calibration_patch_predictions[
    "patch_probability"
].between(0, 1).all()

# ------------------------------------------------------------
# 7. Aggregate to one score per source image
# ------------------------------------------------------------

assert (
    calibration_patch_predictions
    .groupby("filename")["label"]
    .nunique()
    .max()
) == 1

assert (
    calibration_patch_predictions
    .groupby("filename")["patient_id"]
    .nunique()
    .max()
) == 1

calibration_image_predictions = (
    calibration_patch_predictions
    .groupby("filename", as_index=False)
    .agg(
        patient_id=("patient_id", "first"),
        label=("label", "first"),
        image_logit=("patch_logit", "mean"),
        mean_patch_probability=(
            "patch_probability", "mean"
        ),
        n_patches=("patch_index", "size")
    )
)

# PRIMARY PROTOCOL:
# sigmoid(mean patch logit)

calibration_image_predictions["probability"] = (
    torch.sigmoid(
        torch.tensor(
            calibration_image_predictions[
                "image_logit"
            ].to_numpy(),
            dtype=torch.float64
        )
    )
    .numpy()
)

calibration_image_predictions["prediction"] = (
    calibration_image_predictions["probability"]
    .ge(0.5)
    .astype(int)
)

assert len(calibration_image_predictions) == 382

assert (
    calibration_image_predictions["patient_id"]
    .nunique()
    == 16
)

assert (
    calibration_image_predictions["n_patches"]
    .sum()
    == 6079
)

assert calibration_image_predictions[
    "n_patches"
].between(1, 16).all()

assert calibration_image_predictions[
    "probability"
].between(0, 1).all()

assert np.isfinite(
    calibration_image_predictions["image_logit"]
).all()

assert set(
    calibration_image_predictions["patient_id"]
) == calibration_patient_ids

assert (
    calibration_image_predictions["label"]
    .value_counts()
    .to_dict()
    == {0: 121, 1: 261}
)

print("PASS: Exactly 382 image-level predictions obtained.")
print("PASS: Mean-logit primary aggregation verified.")
print("PASS: Image labels and patient membership verified.")

# ------------------------------------------------------------
# 8. Verify CNN weights remained unchanged
# ------------------------------------------------------------

for name, tensor in calibration_model.state_dict().items():

    assert torch.equal(
        tensor.detach().cpu(),
        model_state_before[name]
    ), f"Model state changed during inference: {name}"

assert not calibration_model.training

assert all(
    not p.requires_grad
    for p in calibration_model.parameters()
)

print("PASS: Every CNN state tensor remained unchanged.")

del model_state_before

# ------------------------------------------------------------
# 9. Save patch- and image-level predictions
# ------------------------------------------------------------

CALIBRATION_PATCH_LOGITS_PATH = (
    CALIBRATION_RESULTS_DIR /
    "calibration_patch_logits.csv"
)

CALIBRATION_IMAGE_LOGITS_PATH = (
    CALIBRATION_RESULTS_DIR /
    "calibration_image_predictions_uncalibrated.csv"
)

calibration_patch_predictions.to_csv(
    CALIBRATION_PATCH_LOGITS_PATH,
    index=False,
    float_format="%.17g"
)

calibration_image_predictions.to_csv(
    CALIBRATION_IMAGE_LOGITS_PATH,
    index=False,
    float_format="%.17g"
)

assert CALIBRATION_PATCH_LOGITS_PATH.is_file()
assert CALIBRATION_IMAGE_LOGITS_PATH.is_file()

# Reload and verify saved row counts.

saved_patch_predictions = pd.read_csv(
    CALIBRATION_PATCH_LOGITS_PATH
)

saved_image_predictions = pd.read_csv(
    CALIBRATION_IMAGE_LOGITS_PATH
)

assert len(saved_patch_predictions) == 6079
assert len(saved_image_predictions) == 382

# ------------------------------------------------------------
# 10. Save inference provenance
# ------------------------------------------------------------

CALIBRATION_INFERENCE_AUDIT_PATH = (
    CALIBRATION_RESULTS_DIR /
    "calibration_inference_audit.json"
)

inference_audit = {
    "stage": 7,
    "architecture": "efficientnet_b0",
    "checkpoint_epoch": 5,
    "checkpoint_sha256":
        EXPECTED_CHECKPOINT_SHA256,
    "cohort": "Calibration",
    "magnification": "40x",
    "patients": 16,
    "images": 382,
    "patches": 6079,
    "inference_batch_size":
        CALIBRATION_BATCH_SIZE,
    "inference_precision": "float32",
    "random_augmentation": False,
    "imagenet_normalization": True,
    "aggregation": "mean_patch_logit",
    "primary_probability":
        "sigmoid(mean_patch_logit)",
    "alternate_probability":
        "mean(sigmoid(patch_logit))",
    "manifest_sha256":
        stage7_sha256(CALIBRATION_MANIFEST_PATH),
    "patch_predictions_sha256":
        stage7_sha256(CALIBRATION_PATCH_LOGITS_PATH),
    "image_predictions_sha256":
        stage7_sha256(CALIBRATION_IMAGE_LOGITS_PATH),
    "model_state_unchanged": True,
    "calibrators_fitted": False,
    "test_data_used": False,
    "external_data_used": False
}

with open(
    CALIBRATION_INFERENCE_AUDIT_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(inference_audit, f, indent=2)

# ------------------------------------------------------------
# 11. Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STAGE 7 — CALIBRATION INFERENCE SUMMARY")
print("=" * 70)

print("Calibration patients:", 16)
print("Calibration images:", 382)
print("Calibration patches:", 6079)
print("Benign images:", 121)
print("Malignant images:", 261)

print("\nImage-level prediction preview:")
display(
    calibration_image_predictions.head(5)
)

print("\nPrimary image-level probability summary:")
print(
    calibration_image_predictions[
        "probability"
    ].describe().to_string()
)

print("\n" + "=" * 70)
print("STAGE 7 — CELL 9 FINAL AUDIT")
print("=" * 70)

print("PASS: Original patch dataset reused.")
print("PASS: Deterministic inference preprocessing applied.")
print("PASS: All 6,079 Calibration patches processed.")
print("PASS: Patch predictions aligned to source manifest.")
print("PASS: Exactly 382 image predictions generated.")
print("PASS: Original mean-logit aggregation preserved.")
print("PASS: Patch-level prediction CSV saved.")
print("PASS: Image-level prediction CSV saved.")
print("PASS: Inference SHA-256 hashes recorded.")
print("PASS: All CNN weights and buffers unchanged.")
print("PASS: No calibrators fitted.")
print("PASS: No Test or external data accessed.")

print("\nImage predictions:", CALIBRATION_IMAGE_LOGITS_PATH)
print("Inference audit:", CALIBRATION_INFERENCE_AUDIT_PATH)


PASS: Calibration manifest verified.
PASS: Final Epoch 5 model identity verified.
PASS: All CNN parameters frozen.
PASS: Original BreakHisPatchDataset restored.
PASS: Deterministic inference transform prepared.
PASS: 6,079 patches loaded into inference dataset.
PASS: DataLoader configured without shuffling.

CALIBRATION PATCH INFERENCE


Frozen EfficientNet-B0 inference:   0%|          | 0/190 [00:00<?, ?it/s]


PASS: Inference completed for 6,079 patches.
PASS: Every prediction matches its original patch.
PASS: Patient, filename, patch index and label aligned.
PASS: Exactly 382 image-level predictions obtained.
PASS: Mean-logit primary aggregation verified.
PASS: Image labels and patient membership verified.
PASS: Every CNN state tensor remained unchanged.

STAGE 7 — CALIBRATION INFERENCE SUMMARY
Calibration patients: 16
Calibration images: 382
Calibration patches: 6079
Benign images: 121
Malignant images: 261

Image-level prediction preview:


,filename,patient_id,label,image_logit,mean_patch_probability,n_patches,probability,prediction
0,SOB_B_A-14-22549G-40-001.png,14-22549G,0,5.512064,0.832057,16,0.995978,1
1,SOB_B_A-14-22549G-40-002.png,14-22549G,0,4.305277,0.851788,16,0.986683,1
2,SOB_B_A-14-22549G-40-003.png,14-22549G,0,6.476140,0.963304,16,0.998463,1
3,SOB_B_A-14-22549G-40-004.png,14-22549G,0,9.555709,0.999651,16,0.999929,1
4,SOB_B_A-14-22549G-40-005.png,14-22549G,0,5.207844,0.886965,16,0.994556,1



Primary image-level probability summary:
count    382.000000
mean       0.768541
std        0.393978
min        0.000005
25%        0.743064
50%        0.999884
75%        1.000000
max        1.000000

STAGE 7 — CELL 9 FINAL AUDIT
PASS: Original patch dataset reused.
PASS: Deterministic inference preprocessing applied.
PASS: All 6,079 Calibration patches processed.
PASS: Patch predictions aligned to source manifest.
PASS: Exactly 382 image predictions generated.
PASS: Original mean-logit aggregation preserved.
PASS: Patch-level prediction CSV saved.
PASS: Image-level prediction CSV saved.
PASS: Inference SHA-256 hashes recorded.
PASS: All CNN weights and buffers unchanged.
PASS: No calibrators fitted.
PASS: No Test or external data accessed.

Image predictions: /content/breast-histology-calibration/results/posthoc_calibration/efficientnet_b0/calibration_image_predictions_uncalibrated.csv
Inference audit: /content/breast-histology-calibration/results/posthoc_calibration/efficientnet_b0

In [ ]:

# ============================================================
# Fit Temperature Scaling and Platt Scaling
# Using Image-Level Calibration Logits Only
# ============================================================

from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit

# ------------------------------------------------------------
# 1. Verify the saved Calibration predictions
# ------------------------------------------------------------

assert CALIBRATION_IMAGE_LOGITS_PATH.is_file()

assert stage7_sha256(
    CALIBRATION_MANIFEST_PATH
) == (
    "249bc344b2e6a286c6d0968b292c36b479a1325e83752fcc2d0493bbfe41cce5"
)

assert stage7_sha256(
    CHECKPOINT_PATH
) == EXPECTED_CHECKPOINT_SHA256

assert not calibration_model.training

assert all(
    not p.requires_grad
    for p in calibration_model.parameters()
)

cal_fit_images = pd.read_csv(
    CALIBRATION_IMAGE_LOGITS_PATH,
    dtype={
        "patient_id": str,
        "filename": str
    }
)

assert len(cal_fit_images) == 382
assert cal_fit_images["filename"].is_unique
assert cal_fit_images["patient_id"].nunique() == 16

assert set(cal_fit_images["patient_id"]) == (
    calibration_patient_ids
)

assert cal_fit_images["label"].value_counts().to_dict() == {
    0: 121,
    1: 261
}

assert cal_fit_images["n_patches"].sum() == 6079

z = cal_fit_images["image_logit"].to_numpy(
    dtype=np.float64
)

y = cal_fit_images["label"].to_numpy(
    dtype=np.int64
)

assert np.isfinite(z).all()
assert set(np.unique(y)) == {0, 1}

assert np.std(z) > 0

np.testing.assert_allclose(
    expit(z),
    cal_fit_images["probability"].to_numpy(),
    rtol=0,
    atol=1e-12
)

print("=" * 70)
print("CALIBRATION FITTING — INPUT AUDIT")
print("=" * 70)

print("Images:", len(y))
print("Patients:", cal_fit_images["patient_id"].nunique())
print("Benign:", int((y == 0).sum()))
print("Malignant:", int((y == 1).sum()))
print("Observed malignant fraction:", round(y.mean(), 6))
print("Mean uncalibrated probability:", round(expit(z).mean(), 6))

print("\nPASS: Original Calibration image logits verified.")
print("PASS: Final model checkpoint verified.")
print("PASS: No Test data used.")

# ------------------------------------------------------------
# 2. Define stable image-level negative log-likelihood
# ------------------------------------------------------------

def binary_nll_from_logits(logits, labels):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    labels = np.asarray(
        labels,
        dtype=np.float64
    )

    # Numerically stable binary cross-entropy.
    return float(
        np.mean(
            np.logaddexp(0.0, logits)
            - labels * logits
        )
    )


# ------------------------------------------------------------
# 3. Fit Temperature Scaling
#
# p = sigmoid(image_logit / T), with T > 0.
#
# Log-temperature optimization guarantees T is positive.
# Temperature scaling does not change logit ranking or
# the zero-logit / 0.5 decision boundary.
# ------------------------------------------------------------

TEMP_LOWER = 0.05
TEMP_UPPER = 100.0

def temperature_objective(log_temperature):

    temperature = np.exp(log_temperature)

    return binary_nll_from_logits(
        z / temperature,
        y
    )


temperature_result = minimize_scalar(
    temperature_objective,
    bounds=(
        np.log(TEMP_LOWER),
        np.log(TEMP_UPPER)
    ),
    method="bounded",
    options={
        "xatol": 1e-11,
        "maxiter": 2000
    }
)

assert temperature_result.success, (
    temperature_result.message
)

TEMPERATURE = float(
    np.exp(temperature_result.x)
)

assert np.isfinite(TEMPERATURE)
assert TEMP_LOWER <= TEMPERATURE <= TEMP_UPPER

temperature_logits = z / TEMPERATURE

temperature_probs = expit(
    temperature_logits
)

assert np.isfinite(temperature_probs).all()

assert np.array_equal(
    z >= 0,
    temperature_logits >= 0
)

print("\n" + "=" * 70)
print("TEMPERATURE SCALING")
print("=" * 70)

print("Fitted temperature:", round(TEMPERATURE, 8))

if TEMPERATURE > 1:
    print(
        "Interpretation: Temperature > 1 softens "
        "the model's confidence."
    )
elif TEMPERATURE < 1:
    print(
        "Interpretation: Temperature < 1 sharpens "
        "the model's confidence."
    )
else:
    print("Interpretation: No confidence rescaling.")

print("PASS: Positive temperature fitted.")
print("PASS: Original decision boundary preserved.")

# ------------------------------------------------------------
# 4. Fit Platt Scaling
#
# p = sigmoid(a * image_logit + b)
#
# Optimize unregularized binomial NLL.
# Standardize logits only for numerical optimization;
# convert parameters back to the ORIGINAL logit scale.
# ------------------------------------------------------------

logit_mean = float(np.mean(z))
logit_std = float(np.std(z))

assert logit_std > 0

z_standardized = (
    z - logit_mean
) / logit_std


def platt_objective_and_gradient(params):

    alpha, beta = params

    transformed_logits = (
        alpha * z_standardized + beta
    )

    loss = binary_nll_from_logits(
        transformed_logits,
        y
    )

    residual = (
        expit(transformed_logits) - y
    )

    gradient = np.array([
        np.mean(residual * z_standardized),
        np.mean(residual)
    ], dtype=np.float64)

    return loss, gradient


# Initialize Platt scaling from the fitted
# temperature transformation.
initial_alpha = (
    logit_std / TEMPERATURE
)

initial_beta = (
    logit_mean / TEMPERATURE
)

platt_result = minimize(
    platt_objective_and_gradient,
    x0=np.array([
        initial_alpha,
        initial_beta
    ]),
    method="L-BFGS-B",
    jac=True,
    options={
        "maxiter": 5000,
        "ftol": 1e-12,
        "gtol": 1e-9
    }
)

assert platt_result.success, (
    f"Platt optimization did not converge: "
    f"{platt_result.message}"
)

alpha, beta = (
    platt_result.x.astype(np.float64)
)

# Transform parameters to unstandardized image logits.
PLATT_SLOPE = float(
    alpha / logit_std
)

PLATT_INTERCEPT = float(
    beta - PLATT_SLOPE * logit_mean
)

assert np.isfinite(PLATT_SLOPE)
assert np.isfinite(PLATT_INTERCEPT)

platt_logits = (
    PLATT_SLOPE * z + PLATT_INTERCEPT
)

platt_probs = expit(
    platt_logits
)

np.testing.assert_allclose(
    platt_logits,
    alpha * z_standardized + beta,
    atol=1e-10,
    rtol=1e-10
)

print("\n" + "=" * 70)
print("PLATT SCALING")
print("=" * 70)

print("Fitted slope:", round(PLATT_SLOPE, 8))
print("Fitted intercept:", round(PLATT_INTERCEPT, 8))

if PLATT_SLOPE <= 0:

    print(
        "WARNING: Non-positive Platt slope. "
        "This can reverse or collapse the original "
        "prediction ranking. Review this result."
    )

else:

    print(
        "PASS: Positive Platt slope; "
        "prediction ranking preserved."
    )

print("PASS: Platt scaling optimization completed.")
print("PASS: Parameters expressed on original logit scale.")

# ------------------------------------------------------------
# 5. Descriptive Calibration-cohort metrics
#
# IMPORTANT:
# These are IN-SAMPLE fitting diagnostics,
# not independent generalization estimates.
# ------------------------------------------------------------

def ece10_positive_class(labels, probabilities):

    # Ten equal-width bins over malignant probability.
    # Compare mean predicted malignant probability
    # against observed malignant frequency.

    edges = np.linspace(0.0, 1.0, 11)

    result = 0.0
    n = len(labels)

    for index in range(10):

        lower = edges[index]
        upper = edges[index + 1]

        if index == 9:
            mask = (
                (probabilities >= lower)
                & (probabilities <= upper)
            )
        else:
            mask = (
                (probabilities >= lower)
                & (probabilities < upper)
            )

        if not np.any(mask):
            continue

        bin_predicted = float(
            np.mean(probabilities[mask])
        )

        bin_observed = float(
            np.mean(labels[mask])
        )

        result += (
            np.sum(mask) / n
        ) * abs(
            bin_predicted - bin_observed
        )

    return float(result)


def fitting_metrics(method_name, logits):

    probs = expit(logits)

    predictions = (
        probs >= 0.5
    ).astype(int)

    benign_mask = y == 0
    malignant_mask = y == 1

    specificity = np.mean(
        predictions[benign_mask] == 0
    )

    sensitivity = np.mean(
        predictions[malignant_mask] == 1
    )

    return {
        "method": method_name,
        "nll": binary_nll_from_logits(logits, y),
        "brier": float(
            np.mean((probs - y) ** 2)
        ),
        "ece10": ece10_positive_class(
            y, probs
        ),
        "balanced_accuracy": float(
            (specificity + sensitivity) / 2
        ),
        "mean_predicted_malignant_probability": float(
            np.mean(probs)
        ),
        "observed_malignant_fraction": float(
            np.mean(y)
        )
    }


calibration_fit_metrics = pd.DataFrame([
    fitting_metrics(
        "Uncalibrated",
        z
    ),
    fitting_metrics(
        "Temperature scaling",
        temperature_logits
    ),
    fitting_metrics(
        "Platt scaling",
        platt_logits
    )
])

# Each fitted family contains the identity / temperature
# model, so optimized in-sample NLL should not increase.
baseline_nll = calibration_fit_metrics.iloc[0]["nll"]
temperature_nll = calibration_fit_metrics.iloc[1]["nll"]
platt_nll = calibration_fit_metrics.iloc[2]["nll"]

assert temperature_nll <= baseline_nll + 1e-8

assert platt_nll <= temperature_nll + 1e-7

print("\n" + "=" * 70)
print("CALIBRATION-COHORT FITTING DIAGNOSTICS")
print("=" * 70)

display(calibration_fit_metrics.round(6))

print(
    "\nNOTE: These metrics are calculated on the "
    "same 382 images used to fit the calibrators."
)

print(
    "They cannot establish improvement on "
    "independent patients or distribution shifts."
)

# ------------------------------------------------------------
# 6. Save all image-level calibrated predictions
# ------------------------------------------------------------

calibration_all_methods = (
    cal_fit_images.copy()
)

calibration_all_methods[
    "temperature_probability"
] = temperature_probs

calibration_all_methods[
    "platt_probability"
] = platt_probs

calibration_all_methods[
    "temperature_prediction"
] = (temperature_probs >= 0.5).astype(int)

calibration_all_methods[
    "platt_prediction"
] = (platt_probs >= 0.5).astype(int)

assert len(calibration_all_methods) == 382

assert calibration_all_methods[
    "temperature_probability"
].between(0, 1).all()

assert calibration_all_methods[
    "platt_probability"
].between(0, 1).all()

CALIBRATION_ALL_METHODS_PATH = (
    CALIBRATION_RESULTS_DIR /
    "calibration_image_predictions_all_methods.csv"
)

CALIBRATION_FIT_DIAGNOSTICS_PATH = (
    CALIBRATION_RESULTS_DIR /
    "calibration_fitting_diagnostics.csv"
)

calibration_all_methods.to_csv(
    CALIBRATION_ALL_METHODS_PATH,
    index=False,
    float_format="%.17g"
)

calibration_fit_metrics.to_csv(
    CALIBRATION_FIT_DIAGNOSTICS_PATH,
    index=False,
    float_format="%.17g"
)

# ------------------------------------------------------------
# 7. Save the fitted parameters and provenance
# ------------------------------------------------------------

CALIBRATION_PARAMETERS_PATH = (
    CALIBRATION_RESULTS_DIR /
    "calibration_fitted_parameters.json"
)

parameters_record = {
    "stage": 7,
    "architecture": "efficientnet_b0",
    "final_checkpoint_epoch": 5,
    "final_checkpoint_sha256":
        EXPECTED_CHECKPOINT_SHA256,
    "fitting_cohort": "Calibration",
    "magnification": "40x",
    "patients": 16,
    "images": 382,
    "benign_images": 121,
    "malignant_images": 261,
    "unit_of_calibration": "source_image",
    "sample_weighting": "equal_weight_per_image",
    "primary_aggregation": "mean_patch_logit",
    "calibration_manifest_sha256":
        stage7_sha256(CALIBRATION_MANIFEST_PATH),
    "input_image_logits_sha256":
        stage7_sha256(CALIBRATION_IMAGE_LOGITS_PATH),
    "temperature_scaling": {
        "formula": "sigmoid(image_logit / temperature)",
        "temperature": TEMPERATURE,
        "optimization": "bounded_minimize_scalar_log_temperature",
        "temperature_lower_bound": TEMP_LOWER,
        "temperature_upper_bound": TEMP_UPPER,
        "objective": "mean_image_binary_nll",
        "optimizer_success": bool(
            temperature_result.success
        )
    },
    "platt_scaling": {
        "formula": "sigmoid(slope * image_logit + intercept)",
        "slope": PLATT_SLOPE,
        "intercept": PLATT_INTERCEPT,
        "regularization": "none",
        "optimization": "L-BFGS-B",
        "objective": "mean_image_binary_nll",
        "optimizer_success": bool(
            platt_result.success
        )
    },
    "ece10_definition":
        "10 equal-width positive-class probability bins",
    "diagnostics_are_in_sample": True,
    "cnn_weights_updated": False,
    "test_data_used": False,
    "external_data_used": False
}

with open(
    CALIBRATION_PARAMETERS_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        parameters_record,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 8. Verify all saved fitting records
# ------------------------------------------------------------

assert CALIBRATION_ALL_METHODS_PATH.is_file()
assert CALIBRATION_FIT_DIAGNOSTICS_PATH.is_file()
assert CALIBRATION_PARAMETERS_PATH.is_file()

with open(
    CALIBRATION_PARAMETERS_PATH,
    encoding="utf-8"
) as f:
    saved_parameters = json.load(f)

assert saved_parameters[
    "temperature_scaling"
]["temperature"] == TEMPERATURE

assert saved_parameters[
    "platt_scaling"
]["slope"] == PLATT_SLOPE

assert saved_parameters[
    "platt_scaling"
]["intercept"] == PLATT_INTERCEPT

assert saved_parameters[
    "input_image_logits_sha256"
] == stage7_sha256(
    CALIBRATION_IMAGE_LOGITS_PATH
)

assert len(
    pd.read_csv(CALIBRATION_ALL_METHODS_PATH)
) == 382

assert len(
    pd.read_csv(CALIBRATION_FIT_DIAGNOSTICS_PATH)
) == 3

assert not calibration_model.training

assert all(
    not p.requires_grad
    for p in calibration_model.parameters()
)

print("\n" + "=" * 70)
print("STAGE 7 — CELL 10 FINAL AUDIT")
print("=" * 70)

print("PASS: Temperature scaling fitted.")
print("PASS: Platt scaling fitted.")
print("PASS: Both calibrators use image-level logits.")
print("PASS: No CNN inference repeated.")
print("PASS: Descriptive fitting diagnostics generated.")
print("PASS: Calibrated image predictions saved.")
print("PASS: Calibration parameters saved and reloaded.")
print("PASS: Input and model SHA-256 provenance recorded.")
print("PASS: Final EfficientNet-B0 remains frozen.")
print("PASS: No Test or external data accessed.")

print("\nTemperature:", round(TEMPERATURE, 8))
print("Platt slope:", round(PLATT_SLOPE, 8))
print("Platt intercept:", round(PLATT_INTERCEPT, 8))

print("\nParameters:", CALIBRATION_PARAMETERS_PATH)
print("Diagnostics:", CALIBRATION_FIT_DIAGNOSTICS_PATH)


CALIBRATION FITTING — INPUT AUDIT
Images: 382
Patients: 16
Benign: 121
Malignant: 261
Observed malignant fraction: 0.683246
Mean uncalibrated probability: 0.768541

PASS: Original Calibration image logits verified.
PASS: Final model checkpoint verified.
PASS: No Test data used.

TEMPERATURE SCALING
Fitted temperature: 4.06919314
Interpretation: Temperature > 1 softens the model's confidence.
PASS: Positive temperature fitted.
PASS: Original decision boundary preserved.

PLATT SCALING
Fitted slope: 0.29141657
Fitted intercept: -0.70965901
PASS: Positive Platt slope; prediction ranking preserved.
PASS: Platt scaling optimization completed.
PASS: Parameters expressed on original logit scale.

CALIBRATION-COHORT FITTING DIAGNOSTICS


,method,nll,brier,ece10,balanced_accuracy,mean_predicted_malignant_probability,observed_malignant_fraction
0,Uncalibrated,0.669298,0.138768,0.136185,0.772917,0.768541,0.683246
1,Temperature scaling,0.313066,0.103280,0.085776,0.772917,0.736718,0.683246
2,Platt scaling,0.295131,0.099332,0.078814,0.797600,0.683246,0.683246



NOTE: These metrics are calculated on the same 382 images used to fit the calibrators.
They cannot establish improvement on independent patients or distribution shifts.

STAGE 7 — CELL 10 FINAL AUDIT
PASS: Temperature scaling fitted.
PASS: Platt scaling fitted.
PASS: Both calibrators use image-level logits.
PASS: No CNN inference repeated.
PASS: Descriptive fitting diagnostics generated.
PASS: Calibrated image predictions saved.
PASS: Calibration parameters saved and reloaded.
PASS: Input and model SHA-256 provenance recorded.
PASS: Final EfficientNet-B0 remains frozen.
PASS: No Test or external data accessed.

Temperature: 4.06919314
Platt slope: 0.29141657
Platt intercept: -0.70965901

Parameters: /content/breast-histology-calibration/results/posthoc_calibration/efficientnet_b0/calibration_fitted_parameters.json
Diagnostics: /content/breast-histology-calibration/results/posthoc_calibration/efficientnet_b0/calibration_fitting_diagnostics.csv


In [ ]:

# ============================================================
# Lock Calibrators, Verify Results, and Export Stage 7
# ============================================================

# No new imports needed.

# ------------------------------------------------------------
# 1. Reload fitted parameters and saved predictions
# ------------------------------------------------------------

with open(
    CALIBRATION_PARAMETERS_PATH,
    encoding="utf-8"
) as f:
    locked_parameters = json.load(f)

image_inputs = pd.read_csv(
    CALIBRATION_IMAGE_LOGITS_PATH,
    dtype={"patient_id": str, "filename": str}
)

all_methods = pd.read_csv(
    CALIBRATION_ALL_METHODS_PATH,
    dtype={"patient_id": str, "filename": str}
)

fit_diagnostics = pd.read_csv(
    CALIBRATION_FIT_DIAGNOSTICS_PATH
)

assert len(image_inputs) == 382
assert len(all_methods) == 382
assert len(fit_diagnostics) == 3

assert image_inputs["filename"].is_unique
assert all_methods["filename"].is_unique

assert image_inputs["filename"].tolist() == (
    all_methods["filename"].tolist()
)

assert set(all_methods["patient_id"]) == (
    calibration_patient_ids
)

assert set(all_methods["patient_id"]).isdisjoint(
    development_patient_ids | test_patient_ids
)

assert all_methods["label"].value_counts().to_dict() == {
    0: 121,
    1: 261
}

print("PASS: Calibration records independently reloaded.")
print("PASS: Frozen patient membership verified.")

# ------------------------------------------------------------
# 2. Verify the locked calibration formulas
# ------------------------------------------------------------

locked_temperature = float(
    locked_parameters["temperature_scaling"]["temperature"]
)

locked_slope = float(
    locked_parameters["platt_scaling"]["slope"]
)

locked_intercept = float(
    locked_parameters["platt_scaling"]["intercept"]
)

assert locked_temperature > 0
assert locked_slope > 0

image_logits = image_inputs[
    "image_logit"
].to_numpy(dtype=np.float64)

expected_uncalibrated = expit(image_logits)

expected_temperature = expit(
    image_logits / locked_temperature
)

expected_platt = expit(
    locked_slope * image_logits + locked_intercept
)

for column, expected in [
    ("probability", expected_uncalibrated),
    ("temperature_probability", expected_temperature),
    ("platt_probability", expected_platt)
]:
    np.testing.assert_allclose(
        all_methods[column].to_numpy(
            dtype=np.float64
        ),
        expected,
        rtol=0,
        atol=1e-12
    )

# Temperature scaling must preserve the 0.5 decisions.
assert np.array_equal(
    expected_uncalibrated >= 0.5,
    expected_temperature >= 0.5
)

# Positive Platt slope must preserve original ranking.
assert locked_slope > 0

print("PASS: Both calibrator formulas independently verified.")
print("PASS: Temperature decision boundary preserved.")
print("PASS: Platt slope preserves logit ranking.")

# ------------------------------------------------------------
# 3. Verify CNN checkpoint and live model integrity
# ------------------------------------------------------------

assert stage7_sha256(
    CHECKPOINT_PATH
) == EXPECTED_CHECKPOINT_SHA256

assert not calibration_model.training

assert all(
    not p.requires_grad
    for p in calibration_model.parameters()
)

verified_checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

assert verified_checkpoint["epoch_completed"] == 5

for name, saved_tensor in verified_checkpoint[
    "model_state_dict"
].items():

    live_tensor = calibration_model.state_dict()[name]

    assert torch.equal(
        saved_tensor.cpu(),
        live_tensor.detach().cpu()
    ), f"Model tensor changed: {name}"

del verified_checkpoint

print("PASS: Live CNN exactly matches the final checkpoint.")
print("PASS: No CNN weights or buffers changed.")

# ------------------------------------------------------------
# 4. Verify fitting provenance
# ------------------------------------------------------------

assert locked_parameters[
    "final_checkpoint_sha256"
] == EXPECTED_CHECKPOINT_SHA256

assert locked_parameters[
    "calibration_manifest_sha256"
] == stage7_sha256(
    CALIBRATION_MANIFEST_PATH
)

assert locked_parameters[
    "input_image_logits_sha256"
] == stage7_sha256(
    CALIBRATION_IMAGE_LOGITS_PATH
)

assert locked_parameters[
    "diagnostics_are_in_sample"
] is True

assert locked_parameters["test_data_used"] is False
assert locked_parameters["external_data_used"] is False

assert fit_diagnostics["method"].tolist() == [
    "Uncalibrated",
    "Temperature scaling",
    "Platt scaling"
]

# Verify the fitted NLL values against saved diagnostics.
labels = image_inputs["label"].to_numpy(
    dtype=np.float64
)

for row_index, transformed_logits in enumerate([
    image_logits,
    image_logits / locked_temperature,
    locked_slope * image_logits + locked_intercept
]):

    recalculated_nll = float(
        np.mean(
            np.logaddexp(0.0, transformed_logits)
            - labels * transformed_logits
        )
    )

    assert np.isclose(
        recalculated_nll,
        fit_diagnostics.iloc[row_index]["nll"],
        atol=1e-12,
        rtol=0
    )

print("PASS: Calibration provenance hashes verified.")
print("PASS: All three NLL values independently reproduced.")

# ------------------------------------------------------------
# 5. Save the calibration lock record
# ------------------------------------------------------------

CALIBRATION_LOCK_PATH = (
    CALIBRATION_RESULTS_DIR /
    "calibration_lock_record.json"
)

lock_record = {
    "stage": 7,
    "status": "completed_and_locked",
    "architecture": "efficientnet_b0",
    "final_checkpoint_epoch": 5,
    "final_checkpoint_sha256":
        EXPECTED_CHECKPOINT_SHA256,
    "fitting_partition": "Calibration",
    "fitting_patients": 16,
    "fitting_images": 382,
    "fitting_patches": 6079,
    "aggregation": "mean_patch_logit",
    "calibration_sample_unit": "source_image",
    "temperature_scaling": {
        "temperature": locked_temperature,
        "formula": "sigmoid(image_logit / temperature)"
    },
    "platt_scaling": {
        "slope": locked_slope,
        "intercept": locked_intercept,
        "formula":
            "sigmoid(slope * image_logit + intercept)"
    },
    "calibration_manifest_sha256":
        stage7_sha256(CALIBRATION_MANIFEST_PATH),
    "calibration_image_logits_sha256":
        stage7_sha256(CALIBRATION_IMAGE_LOGITS_PATH),
    "calibration_parameters_sha256":
        stage7_sha256(CALIBRATION_PARAMETERS_PATH),
    "calibration_diagnostics_sha256":
        stage7_sha256(CALIBRATION_FIT_DIAGNOSTICS_PATH),
    "calibration_predictions_sha256":
        stage7_sha256(CALIBRATION_ALL_METHODS_PATH),
    "optimization_objective": "image_level_binary_nll",
    "calibration_results_are_in_sample": True,
    "independent_test_evaluation_completed": False,
    "cnn_weights_updated": False,
    "test_data_used_for_fitting": False,
    "external_data_used_for_fitting": False,
    "calibrator_selection_using_test": False
}

temporary_lock_path = CALIBRATION_LOCK_PATH.with_suffix(
    ".json.tmp"
)

with open(
    temporary_lock_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(lock_record, f, indent=2)

temporary_lock_path.replace(
    CALIBRATION_LOCK_PATH
)

with open(
    CALIBRATION_LOCK_PATH,
    encoding="utf-8"
) as f:
    saved_lock = json.load(f)

assert saved_lock["status"] == "completed_and_locked"

print("PASS: Final calibration lock record saved.")

# ------------------------------------------------------------
# 6. Create one GitHub-ready Stage 7 results ZIP
# ------------------------------------------------------------

STAGE7_RESULTS_ZIP = Path(
    "/content/CIS631_Stage7_Calibration_Results_For_GitHub.zip"
)

# Include the Calibration manifest, fitting records,
# patch and image predictions, and verification audits.
# Do NOT include trained .pt model checkpoints.

stage7_files = [
    CALIBRATION_MANIFEST_PATH
]

stage7_files.extend(
    sorted(
        path
        for path in CALIBRATION_RESULTS_DIR.iterdir()
        if (
            path.is_file()
            and path.suffix.lower() in {".csv", ".json"}
        )
    )
)

assert len(stage7_files) >= 9

for path in stage7_files:
    assert path.is_file()
    assert path.is_relative_to(REPO_ROOT)

with zipfile.ZipFile(
    STAGE7_RESULTS_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for path in stage7_files:

        zf.write(
            path,
            arcname=str(
                path.relative_to(REPO_ROOT)
            )
        )

with zipfile.ZipFile(
    STAGE7_RESULTS_ZIP
) as zf:

    assert zf.testzip() is None

    archive_files = zf.namelist()

    assert len(archive_files) == len(stage7_files)

    assert not any(
        name.lower().endswith((".pt", ".pth", ".ckpt"))
        for name in archive_files
    )

print("PASS: Stage 7 ZIP contents verified.")
print("PASS: No trained model weights included.")

# ------------------------------------------------------------
# 7. Final audit
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STAGE 7 — FINAL CALIBRATION RESULTS")
print("=" * 70)

display(fit_diagnostics.round(6))

print("\nLocked temperature:", locked_temperature)
print("Locked Platt slope:", locked_slope)
print("Locked Platt intercept:", locked_intercept)

print("\n" + "=" * 70)
print("STAGE 7 — COMPLETION AUDIT")
print("=" * 70)

print("PASS: Temperature scaling locked.")
print("PASS: Platt scaling locked.")
print("PASS: Saved probabilities independently reproduced.")
print("PASS: Saved NLL values independently reproduced.")
print("PASS: Final CNN exactly matches verified checkpoint.")
print("PASS: Calibration patient membership verified.")
print("PASS: Calibration parameter hashes recorded.")
print("PASS: Lock record saved.")
print("PASS: GitHub-ready results ZIP verified.")
print("PASS: No model checkpoint included in ZIP.")
print("PASS: No Test or external data used.")
print("PASS: No Test-based calibrator selection.")

print("\nSTAGE 7: COMPLETE AND LOCKED")
print("Results ZIP:", STAGE7_RESULTS_ZIP)
print("Files packaged:", len(stage7_files))

files.download(str(STAGE7_RESULTS_ZIP))


PASS: Calibration records independently reloaded.
PASS: Frozen patient membership verified.
PASS: Both calibrator formulas independently verified.
PASS: Temperature decision boundary preserved.
PASS: Platt slope preserves logit ranking.
PASS: Live CNN exactly matches the final checkpoint.
PASS: No CNN weights or buffers changed.
PASS: Calibration provenance hashes verified.
PASS: All three NLL values independently reproduced.
PASS: Final calibration lock record saved.
PASS: Stage 7 ZIP contents verified.
PASS: No trained model weights included.

STAGE 7 — FINAL CALIBRATION RESULTS


,method,nll,brier,ece10,balanced_accuracy,mean_predicted_malignant_probability,observed_malignant_fraction
0,Uncalibrated,0.669298,0.138768,0.136185,0.772917,0.768541,0.683246
1,Temperature scaling,0.313066,0.103280,0.085776,0.772917,0.736718,0.683246
2,Platt scaling,0.295131,0.099332,0.078814,0.797600,0.683246,0.683246



Locked temperature: 4.069193139210192
Locked Platt slope: 0.29141657172422586
Locked Platt intercept: -0.7096590060187311

STAGE 7 — COMPLETION AUDIT
PASS: Temperature scaling locked.
PASS: Platt scaling locked.
PASS: Saved probabilities independently reproduced.
PASS: Saved NLL values independently reproduced.
PASS: Final CNN exactly matches verified checkpoint.
PASS: Calibration patient membership verified.
PASS: Calibration parameter hashes recorded.
PASS: Lock record saved.
PASS: GitHub-ready results ZIP verified.
PASS: No model checkpoint included in ZIP.
PASS: No Test or external data used.
PASS: No Test-based calibrator selection.

STAGE 7: COMPLETE AND LOCKED
Results ZIP: /content/CIS631_Stage7_Calibration_Results_For_GitHub.zip
Files packaged: 9


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>